In [4]:
import mysql.connector
import sqlite3
from mysql.connector import Error

In [5]:
import mysql.connector
import sqlite3
from mysql.connector import Error
from decimal import Decimal
import datetime
import os
import random

def convert_decimal_to_float(obj):
    """
    Рекурсивно преобразует объекты Decimal в float.
    Также преобразует datetime в строки для SQLite.
    """
    if isinstance(obj, Decimal):
        return float(obj)
    elif isinstance(obj, (datetime.datetime, datetime.date)):
        return str(obj)
    elif isinstance(obj, tuple):
        return tuple(convert_decimal_to_float(item) for item in obj)
    elif isinstance(obj, list):
        return [convert_decimal_to_float(item) for item in obj]
    else:
        return obj

def create_local_db_and_copy_data(db_directory, db_filename='local_hockey_database.db', num_rows=40):
    """
    Создает локальную базу данных SQLite и копирует в нее случайные строки из удаленной MariaDB (Hockey DB).

    :param db_directory: директория, в которой будет храниться локальная база данных
    :param db_filename: имя файла локальной базы данных
    :param num_rows: количество случайных строк для выборки из каждой таблицы
    """
    # Создаем директорию, если она не существует
    os.makedirs(db_directory, exist_ok=True)
    
    # Полный путь к файлу базы данных
    local_db_file = os.path.join(db_directory, db_filename)

    # Удаляем файл базы данных, если он уже существует
    if os.path.exists(local_db_file):
        os.remove(local_db_file)
        print(f"Существующий файл базы данных {local_db_file} удален")

    remote_host = 'relational.fel.cvut.cz'
    remote_port = 3306
    remote_user = 'guest'
    remote_password = 'ctu-relational'
    remote_database = 'Hockey'  # Изменили базу данных на Hockey

    # Подключение к удаленной базе данных
    try:
        remote_conn = mysql.connector.connect(
            host=remote_host,
            port=remote_port,
            user=remote_user,
            password=remote_password,
            database=remote_database
        )
        print("Успешное подключение к удаленной базе данных")
    except Error as e:
        print(f"Ошибка подключения к удаленной базе данных: {e}")
        return

    # Подключение к локальной SQLite базе данных
    try:
        local_conn = sqlite3.connect(local_db_file)
        print(f"Создана локальная база данных SQLite: {local_db_file}")
    except Error as e:
        print(f"Ошибка подключения к локальной базе данных: {e}")
        remote_conn.close()
        return

    cursor_remote = remote_conn.cursor()
    cursor_local = local_conn.cursor()

    # Определяем имена таблиц из базы данных Hockey
    table_names = [
        'AwardsCoaches', 'AwardsMisc', 'AwardsPlayers', 'Coaches', 'CombinedShutouts', 
        'Goalies', 'GoaliesSC', 'GoaliesShootout', 'HOF', 'Master', 'Scoring', 
        'ScoringSC', 'ScoringShootout', 'ScoringSup', 'SeriesPost', 'TeamSplits', 
        'TeamVsTeam', 'Teams', 'TeamsHalf', 'TeamsPost', 'TeamsSC', 'abbrev'
    ]

    # Создаем таблицы в локальной базе данных с внешними ключами
    create_table_sql = {
        'AwardsCoaches': '''
            CREATE TABLE IF NOT EXISTS AwardsCoaches (
                coachID TEXT,
                award TEXT,
                year INTEGER,
                lgID TEXT,
                note TEXT,
                FOREIGN KEY (coachID) REFERENCES Coaches (coachID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'AwardsMisc': '''
            CREATE TABLE IF NOT EXISTS AwardsMisc (
                name TEXT NOT NULL PRIMARY KEY,
                ID TEXT,
                award TEXT,
                year INTEGER,
                lgID TEXT,
                note TEXT
            )
        ''',
        'AwardsPlayers': '''
            CREATE TABLE IF NOT EXISTS AwardsPlayers (
                playerID TEXT NOT NULL,
                award TEXT NOT NULL,
                year INTEGER NOT NULL,
                lgID TEXT,
                note TEXT,
                pos TEXT,
                PRIMARY KEY (playerID, award, year),
                FOREIGN KEY (playerID) REFERENCES Master (playerID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'Coaches': '''
            CREATE TABLE IF NOT EXISTS Coaches (
                coachID TEXT NOT NULL,
                year INTEGER NOT NULL,
                tmID TEXT NOT NULL,
                lgID TEXT,
                stint INTEGER NOT NULL,
                notes TEXT,
                g INTEGER,
                w INTEGER,
                l INTEGER,
                t INTEGER,
                postg TEXT,
                postw TEXT,
                postl TEXT,
                postt TEXT,
                PRIMARY KEY (coachID, year, tmID, stint),
                FOREIGN KEY (year, tmID) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'CombinedShutouts': '''
            CREATE TABLE IF NOT EXISTS CombinedShutouts (
                year INTEGER,
                month INTEGER,
                date INTEGER,
                tmID TEXT,
                oppID TEXT,
                "R/P" TEXT,
                IDgoalie1 TEXT,
                IDgoalie2 TEXT,
                FOREIGN KEY (IDgoalie1) REFERENCES Master (playerID) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (IDgoalie2) REFERENCES Master (playerID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'Goalies': '''
            CREATE TABLE IF NOT EXISTS Goalies (
                playerID TEXT NOT NULL,
                year INTEGER NOT NULL,
                stint INTEGER NOT NULL,
                tmID TEXT,
                lgID TEXT,
                GP TEXT,
                Min TEXT,
                W TEXT,
                L TEXT,
                "T/OL" TEXT,
                ENG TEXT,
                SHO TEXT,
                GA TEXT,
                SA TEXT,
                PostGP TEXT,
                PostMin TEXT,
                PostW TEXT,
                PostL TEXT,
                PostT TEXT,
                PostENG TEXT,
                PostSHO TEXT,
                PostGA TEXT,
                PostSA TEXT,
                PRIMARY KEY (playerID, year, stint),
                FOREIGN KEY (playerID) REFERENCES Master (playerID) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (year, tmID) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'GoaliesSC': '''
            CREATE TABLE IF NOT EXISTS GoaliesSC (
                playerID TEXT NOT NULL,
                year INTEGER NOT NULL,
                tmID TEXT,
                lgID TEXT,
                GP INTEGER,
                Min INTEGER,
                W INTEGER,
                L INTEGER,
                T INTEGER,
                SHO INTEGER,
                GA INTEGER,
                PRIMARY KEY (playerID, year),
                FOREIGN KEY (playerID) REFERENCES Master (playerID) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (year, tmID) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'GoaliesShootout': '''
            CREATE TABLE IF NOT EXISTS GoaliesShootout (
                playerID TEXT,
                year INTEGER,
                stint INTEGER,
                tmID TEXT,
                W INTEGER,
                L INTEGER,
                SA INTEGER,
                GA INTEGER,
                FOREIGN KEY (playerID) REFERENCES Master (playerID) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (year, tmID) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'HOF': '''
            CREATE TABLE IF NOT EXISTS HOF (
                year INTEGER,
                hofID TEXT NOT NULL PRIMARY KEY,
                name TEXT,
                category TEXT
            )
        ''',
        'Master': '''
            CREATE TABLE IF NOT EXISTS Master (
                playerID TEXT,
                coachID TEXT,
                hofID TEXT,
                firstName TEXT,
                lastName TEXT NOT NULL,
                nameNote TEXT,
                nameGiven TEXT,
                nameNick TEXT,
                height TEXT,
                weight TEXT,
                shootCatch TEXT,
                legendsID TEXT,
                ihdbID TEXT,
                hrefID TEXT,
                firstNHL TEXT,
                lastNHL TEXT,
                firstWHA TEXT,
                lastWHA TEXT,
                pos TEXT,
                birthYear TEXT,
                birthMon TEXT,
                birthDay TEXT,
                birthCountry TEXT,
                birthState TEXT,
                birthCity TEXT,
                deathYear TEXT,
                deathMon TEXT,
                deathDay TEXT,
                deathCountry TEXT,
                deathState TEXT,
                deathCity TEXT,
                FOREIGN KEY (coachID) REFERENCES Coaches (coachID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'Scoring': '''
            CREATE TABLE IF NOT EXISTS Scoring (
                playerID TEXT,
                year INTEGER,
                stint INTEGER,
                tmID TEXT,
                lgID TEXT,
                pos TEXT,
                GP INTEGER,
                G INTEGER,
                A INTEGER,
                Pts INTEGER,
                PIM INTEGER,
                "+/-" TEXT,
                PPG TEXT,
                PPA TEXT,
                SHG TEXT,
                SHA TEXT,
                GWG TEXT,
                GTG TEXT,
                SOG TEXT,
                PostGP TEXT,
                PostG TEXT,
                PostA TEXT,
                PostPts TEXT,
                PostPIM TEXT,
                "Post+/-" TEXT,
                PostPPG TEXT,
                PostPPA TEXT,
                PostSHG TEXT,
                PostSHA TEXT,
                PostGWG TEXT,
                PostSOG TEXT,
                FOREIGN KEY (playerID) REFERENCES Master (playerID) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (year, tmID) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'ScoringSC': '''
            CREATE TABLE IF NOT EXISTS ScoringSC (
                playerID TEXT,
                year INTEGER,
                tmID TEXT,
                lgID TEXT,
                pos TEXT,
                GP INTEGER,
                G INTEGER,
                A INTEGER,
                Pts INTEGER,
                PIM INTEGER,
                FOREIGN KEY (playerID) REFERENCES Master (playerID) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (year, tmID) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'ScoringShootout': '''
            CREATE TABLE IF NOT EXISTS ScoringShootout (
                playerID TEXT,
                year INTEGER,
                stint INTEGER,
                tmID TEXT,
                S INTEGER,
                G INTEGER,
                GDG INTEGER,
                FOREIGN KEY (playerID) REFERENCES Master (playerID) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (year, tmID) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'ScoringSup': '''
            CREATE TABLE IF NOT EXISTS ScoringSup (
                playerID TEXT,
                year INTEGER,
                PPA TEXT,
                SHA TEXT,
                FOREIGN KEY (playerID) REFERENCES Master (playerID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'SeriesPost': '''
            CREATE TABLE IF NOT EXISTS SeriesPost (
                year INTEGER,
                round TEXT,
                series TEXT,
                tmIDWinner TEXT,
                lgIDWinner TEXT,
                tmIDLoser TEXT,
                lgIDLoser TEXT,
                W INTEGER,
                L INTEGER,
                T INTEGER,
                GoalsWinner INTEGER,
                GoalsLoser INTEGER,
                note TEXT,
                FOREIGN KEY (year, tmIDWinner) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (year, tmIDLoser) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'TeamSplits': '''
            CREATE TABLE IF NOT EXISTS TeamSplits (
                year INTEGER NOT NULL,
                lgID TEXT,
                tmID TEXT NOT NULL,
                hW INTEGER,
                hL INTEGER,
                hT INTEGER,
                hOTL TEXT,
                rW INTEGER,
                rL INTEGER,
                rT INTEGER,
                rOTL TEXT,
                SepW TEXT,
                SepL TEXT,
                SepT TEXT,
                SepOL TEXT,
                OctW TEXT,
                OctL TEXT,
                OctT TEXT,
                OctOL TEXT,
                NovW TEXT,
                NovL TEXT,
                NovT TEXT,
                NovOL TEXT,
                DecW TEXT,
                DecL TEXT,
                DecT TEXT,
                DecOL TEXT,
                JanW INTEGER,
                JanL INTEGER,
                JanT INTEGER,
                JanOL TEXT,
                FebW INTEGER,
                FebL INTEGER,
                FebT INTEGER,
                FebOL TEXT,
                MarW TEXT,
                MarL TEXT,
                MarT TEXT,
                MarOL TEXT,
                AprW TEXT,
                AprL TEXT,
                AprT TEXT,
                AprOL TEXT,
                PRIMARY KEY (year, tmID),
                FOREIGN KEY (year, tmID) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'TeamVsTeam': '''
            CREATE TABLE IF NOT EXISTS TeamVsTeam (
                year INTEGER NOT NULL,
                lgID TEXT,
                tmID TEXT NOT NULL,
                oppID TEXT NOT NULL,
                W INTEGER,
                L INTEGER,
                T INTEGER,
                OTL TEXT,
                PRIMARY KEY (year, tmID, oppID),
                FOREIGN KEY (year, tmID) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE,
                FOREIGN KEY (oppID, year) REFERENCES Teams (tmID, year) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'Teams': '''
            CREATE TABLE IF NOT EXISTS Teams (
                year INTEGER NOT NULL,
                lgID TEXT,
                tmID TEXT NOT NULL,
                franchID TEXT,
                confID TEXT,
                divID TEXT,
                rank INTEGER,
                playoff TEXT,
                G INTEGER,
                W INTEGER,
                L INTEGER,
                T INTEGER,
                OTL TEXT,
                Pts INTEGER,
                SoW TEXT,
                SoL TEXT,
                GF INTEGER,
                GA INTEGER,
                name TEXT,
                PIM TEXT,
                BenchMinor TEXT,
                PPG TEXT,
                PPC TEXT,
                SHA TEXT,
                PKG TEXT,
                PKC TEXT,
                SHF TEXT,
                PRIMARY KEY (year, tmID)
            )
        ''',
        'TeamsHalf': '''
            CREATE TABLE IF NOT EXISTS TeamsHalf (
                year INTEGER NOT NULL,
                lgID TEXT,
                tmID TEXT NOT NULL,
                half INTEGER NOT NULL,
                rank INTEGER,
                G INTEGER,
                W INTEGER,
                L INTEGER,
                T INTEGER,
                GF INTEGER,
                GA INTEGER,
                PRIMARY KEY (year, tmID, half),
                FOREIGN KEY (tmID, year) REFERENCES Teams (tmID, year) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'TeamsPost': '''
            CREATE TABLE IF NOT EXISTS TeamsPost (
                year INTEGER NOT NULL,
                lgID TEXT,
                tmID TEXT NOT NULL,
                G INTEGER,
                W INTEGER,
                L INTEGER,
                T INTEGER,
                GF INTEGER,
                GA INTEGER,
                PIM TEXT,
                BenchMinor TEXT,
                PPG TEXT,
                PPC TEXT,
                SHA TEXT,
                PKG TEXT,
                PKC TEXT,
                SHF TEXT,
                PRIMARY KEY (year, tmID),
                FOREIGN KEY (year, tmID) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'TeamsSC': '''
            CREATE TABLE IF NOT EXISTS TeamsSC (
                year INTEGER NOT NULL,
                lgID TEXT,
                tmID TEXT NOT NULL,
                G INTEGER,
                W INTEGER,
                L INTEGER,
                T INTEGER,
                GF INTEGER,
                GA INTEGER,
                PIM TEXT,
                PRIMARY KEY (year, tmID),
                FOREIGN KEY (year, tmID) REFERENCES Teams (year, tmID) ON DELETE CASCADE ON UPDATE CASCADE
            )
        ''',
        'abbrev': '''
            CREATE TABLE IF NOT EXISTS abbrev (
                Type TEXT NOT NULL,
                Code TEXT NOT NULL,
                Fullname TEXT,
                PRIMARY KEY (Type, Code)
            )
        '''
    }

    # Создаем все таблицы
    for table_name in table_names:
        cursor_local.execute(create_table_sql[table_name])

    # Отключаем проверку внешних ключей на время вставки данных
    cursor_local.execute("PRAGMA foreign_keys = OFF;")

    # Копируем данные из каждой таблицы (по num_rows случайных строк)
    for table_name in table_names:
        print(f"Копируем данные из таблицы {table_name}...")
        
        # Сначала получаем общее количество строк в таблице
        count_query = f"SELECT COUNT(*) FROM {table_name}"
        cursor_remote.execute(count_query)
        total_rows = cursor_remote.fetchone()[0]
        
        # Если в таблице меньше или столько же строк, сколько нужно выбрать, 
        # просто выбираем все строки
        if total_rows <= num_rows:
            select_query = f"SELECT * FROM {table_name}"
            cursor_remote.execute(select_query)
            rows = cursor_remote.fetchall()
        else:
            # Иначе выбираем случайные строки
            # Получаем все строки
            select_query = f"SELECT * FROM {table_name}"
            cursor_remote.execute(select_query)
            all_rows = cursor_remote.fetchall()
            
            # Выбираем случайные строки
            random_rows = random.sample(all_rows, num_rows)
            rows = random_rows

        # Преобразуем Decimal в float и datetime в строки
        converted_rows = [convert_decimal_to_float(row) for row in rows]

        # Получаем количество столбцов для формирования SQL-запроса
        placeholders = ','.join(['?' for _ in range(len(cursor_remote.description))])
        insert_query = f"INSERT INTO {table_name} VALUES ({placeholders})"

        cursor_local.executemany(insert_query, converted_rows)
        print(f"Скопировано {len(converted_rows)} случайных строк из таблицы {table_name}")

    # Включаем проверку внешних ключей
    cursor_local.execute("PRAGMA foreign_keys = ON;")

    # Сохраняем изменения и закрываем соединения
    local_conn.commit()
    remote_conn.close()
    local_conn.close()
    print(f"Данные успешно скопированы в локальную базу данных: {local_db_file}")

if __name__ == "__main__":
    # Пример использования:
    # Укажите директорию, в которой хотите хранить базу данных
    directory = "."
    create_local_db_and_copy_data(db_directory=directory, num_rows=40)

Существующий файл базы данных ./local_hockey_database.db удален
Успешное подключение к удаленной базе данных
Создана локальная база данных SQLite: ./local_hockey_database.db
Копируем данные из таблицы AwardsCoaches...
Скопировано 40 случайных строк из таблицы AwardsCoaches
Копируем данные из таблицы AwardsMisc...
Скопировано 40 случайных строк из таблицы AwardsMisc
Копируем данные из таблицы AwardsPlayers...
Скопировано 40 случайных строк из таблицы AwardsPlayers
Копируем данные из таблицы Coaches...
Скопировано 40 случайных строк из таблицы Coaches
Копируем данные из таблицы CombinedShutouts...
Скопировано 40 случайных строк из таблицы CombinedShutouts
Копируем данные из таблицы Goalies...
Скопировано 40 случайных строк из таблицы Goalies
Копируем данные из таблицы GoaliesSC...
Скопировано 31 случайных строк из таблицы GoaliesSC
Копируем данные из таблицы GoaliesShootout...
Скопировано 40 случайных строк из таблицы GoaliesShootout
Копируем данные из таблицы HOF...
Скопировано 40 случа

In [13]:
import sqlite3
import pandas as pd
from io import StringIO
import csv

def get_database_info_and_csv():
    # Подключаемся к локальной базе данных
    conn = sqlite3.connect('local_hockey_database.db')
    cursor = conn.cursor()

    # Получаем список таблиц
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    output = StringIO()
    writer = csv.writer(output)

    for table_name in tables:
        table_name = table_name[0]
        print(f"Обработка таблицы: {table_name}")

        # Получаем структуру таблицы (CREATE TABLE)
        cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table_name}';")
        create_table_sql = cursor.fetchone()[0]
        output.write(f"\n--- Структура таблицы {table_name} ---\n")
        output.write(f"{create_table_sql}\n")

        # Получаем и выводим все данные из таблицы
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", conn)
        output.write(f"\n--- Данные таблицы {table_name} ---\n")
        
        # Записываем заголовки
        writer.writerow(df.columns.tolist())
        # Записываем данные
        for row in df.itertuples(index=False, name=None):
            writer.writerow(row)
        output.write("\n")  # Добавляем пустую строку между таблицами

    conn.close()
    
    return output.getvalue()

# Вызов функции и вывод результата
result = get_database_info_and_csv()
print(result)

Обработка таблицы: AwardsCoaches
Обработка таблицы: AwardsMisc
Обработка таблицы: AwardsPlayers
Обработка таблицы: Coaches
Обработка таблицы: CombinedShutouts
Обработка таблицы: Goalies
Обработка таблицы: GoaliesSC
Обработка таблицы: GoaliesShootout
Обработка таблицы: HOF
Обработка таблицы: Master
Обработка таблицы: Scoring
Обработка таблицы: ScoringSC
Обработка таблицы: ScoringShootout
Обработка таблицы: ScoringSup
Обработка таблицы: SeriesPost
Обработка таблицы: TeamSplits
Обработка таблицы: TeamVsTeam
Обработка таблицы: Teams
Обработка таблицы: TeamsHalf
Обработка таблицы: TeamsPost
Обработка таблицы: TeamsSC
Обработка таблицы: abbrev

--- Структура таблицы AwardsCoaches ---
CREATE TABLE AwardsCoaches (
                coachID TEXT,
                award TEXT,
                year INTEGER,
                lgID TEXT,
                note TEXT,
                FOREIGN KEY (coachID) REFERENCES Coaches (coachID) ON DELETE CASCADE ON UPDATE CASCADE
            )

--- Данные таблицы Award

In [10]:
import sqlite3
import pandas as pd

def execute_sql_query(db_file, query):
    """
    Выполняет SQL-запрос к локальной базе данных и возвращает результат в виде DataFrame.

    :param db_file: путь к файлу локальной базы данных SQLite
    :param query: SQL-запрос для выполнения
    :return: результат запроса в виде pandas DataFrame
    """
    # Подключаемся к базе данных
    conn = sqlite3.connect(db_file)
    
    try:
        # Выполняем запрос и возвращаем результат в виде DataFrame
        result_df = pd.read_sql_query(query, conn)
        return result_df
    except Exception as e:
        print(f"Произошла ошибка при выполнении запроса: {e}")
        return None
    finally:
        # Закрываем соединение
        conn.close()

# Пример использования:
# db_file = 'local_credit_database.db'
# query = 'SELECT * FROM member LIMIT 10;'
# result = execute_sql_query(db_file, query)
# print(result)

In [29]:
query = """
SELECT SUM(c.w) AS total_wins_by_jack_adams_coaches
FROM AwardsCoaches ac
JOIN Coaches c ON ac.coachID = c.coachID
WHERE ac.award = 'Jack Adams';
"""

In [30]:
db_file = 'local_hockey_database.db'
result = execute_sql_query(db_file, query)
print(result)

   total_wins_by_jack_adams_coaches
0                               179
